# Week 1, Day 3: CASE WHEN

Exercises: `day-03.md`

In [1]:
import sys
sys.path.append("../..")

import numpy as np
import pandas as pd
import plotly.express as px

from db import connect

con = connect()

## 3.1

In [12]:
con.sql("""
select
payment_type
,count(distinct order_id) as orders_count
,round(sum(payment_value),2) as payment_value_sum
from order_payments
group by all
order by orders_count desc
""").df()

,payment_type,orders_count,payment_value_sum
0,credit_card,76505,12542084.19
1,boleto,19784,2869361.27
2,voucher,3866,379436.87
3,debit_card,1528,217989.79
4,not_defined,3,0.00


## 3.2

In [28]:
con.sql("""
select
--1x, 2-3x, 4-6x, 7x+
case
    when payment_installments == 1 then '1'
    when payment_installments between 2 and 3 then '2-3x'
    when payment_installments between 4 and 6 then '4-6x'
    when payment_installments > 6 then '7x+'
    else 'other'
end as payment_installments_group
,count(order_id) as orders_count
,round(count(order_id) / sum(count(*)) over () * 100,1) as pct_orders
from order_payments
group by all
order by orders_count desc
""").df()

,payment_installments_group,orders_count,pct_orders
0,1,52546,50.6
1,2-3x,22874,22.0
2,4-6x,16257,15.6
3,7x+,12207,11.8
4,other,2,0.0


## 3.3

In [58]:
con.sql("""
select
sum( case
    when order_status in ('delivered') and order_delivered_customer_date is not null then 1
    else 0
end) as 'total_orders'
,sum( case
    when order_status in ('delivered') and order_delivered_customer_date > order_estimated_delivery_date then 1
    else 0
end) as 'late_orders'
,round(sum( case
    when order_status in ('delivered') and order_delivered_customer_date > order_estimated_delivery_date then 1
    else 0
end) / sum( case
    when order_status in ('delivered') and order_delivered_customer_date is not null then 1
    else 0
end) * 100,1) as 'pct_orders'
from orders
group by all
""").df()

,total_orders,late_orders,pct_orders
0,96470.0,7826.0,8.1


## 3.4

In [86]:
df1 = con.sql("""
select * from order_payments
""").df()

#1x, 2-3x, 4-6x, 7x+

df1['bins'] = pd.cut(df1['payment_installments'], bins=[0,1,2,3,4,6,1000])

df1['bins'].value_counts(normalize=True)

bins
(0, 1]       0.505814
(1, 2]       0.119489
(6, 1000]    0.117506
(2, 3]       0.100699
(4, 6]       0.088166
(3, 4]       0.068326
Name: proportion, dtype: float64

## 3.5

In [ ]:
df2 = con.sql("""
select * from orders
""").df()

df2['month_start'] = pd.to_datetime(df2['order_delivered_customer_date']).dt.to_period('M')

df2['is_late'] = np.where(df2['order_delivered_customer_date'] > df2['order_estimated_delivery_date'], 1, 0)

df_agg = df2.groupby(['month_start','is_late'],as_index=False).agg(orders_count=('order_id','count'))

df2_pivot = df_agg.pivot(index='month_start', columns='is_late', values='orders_count').fillna(0)

df2_pivot['total_orders'] = df2_pivot[0] + df2_pivot[1] 
df2_pivot['late_share'] = round(df2_pivot[1] / df2_pivot['total_orders'] * 100,1)

df2_pivot

is_late,0,1,total_orders,late_share
month_start,,,,
2016-10,208.0,0.0,208.0,0.0
2016-11,58.0,2.0,60.0,3.3
2016-12,2.0,2.0,4.0,50.0
2017-01,283.0,0.0,283.0,0.0
2017-02,1349.0,2.0,1351.0,0.1
2017-03,2318.0,64.0,2382.0,2.7
2017-04,1746.0,103.0,1849.0,5.6
2017-05,3525.0,226.0,3751.0,6.0
2017-06,3109.0,114.0,3223.0,3.5
